In [2]:
# -*- coding: utf-8 -*-
"""
Created on Wed Jul 15 10:08:31 2020

@author: PristerM
"""

from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import datetime
from time import sleep
import os
regulatorName = 'BI BRB'
print("running  BI BRB webscaper")
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

now=datetime.datetime.now()
filename= 'BI BRB SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer=pd.ExcelWriter(filename)

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
  'Phone - Mother company': [], 'Check': []}  
regdict={"BI BRB 1":["case 1","https://www.brb.bi/fr/content/syst%C3%A8me-financier-burundais"],
"BI BRB 2":["case 2","https://www.brb.bi/fr/content/emf-en-activit%C3%A9"]
}

cities=['Bubanza', 'Buhongo', 'Bujumbura', 'Bukirasazi', 'Bururi', 'Cankuzo', 'Cibitoke', 'Gitega', 'Kabezi', 'Karuzi', 'Kayanza', 'Kayero', 'Kayogoro', 'Kibondo', 'Kirundo', 'Kisozi', 'Luhwa', 'Makamba', 'Magara', 'Mukenke', 'Muramvya', 'Murore', 'Musenyi', 'Muyaga', 'Muyinga', 'Mwaro', 'Ngozi', 'Nyanza-Lac', 'Rugari', 'Rumonge', 'Rutana', 'Ruyigi', 'Zanandore']
                
driver = webdriver.Chrome() 
driver.maximize_window()
process_date=now.strftime('%Y-%m-%d')
for reg in regdict:
    print(f"working with {reg}")
    driver.get(regdict[reg][1])
    sleep(5)
    soup = BeautifulSoup(driver.page_source.replace("<strong>","<strong>***"),"html.parser")
    if regdict[reg][0] == "case 1":
        div = soup.find("div",{"id":"block-system-main"})
        ps = div.find_all("p")
        ps = list(filter(lambda x: len(x.text.strip())>0,ps))
        #for p in ps:
            #print(p.text)
        indlist = []
        for ind in range(len(ps)):
            if "***" in ps[ind].text:
                indlist.append(ind).replace("***","")
        print(indlist)
        banklist = []
        for ind in range(len(indlist)-1):
            if indlist[ind+1]-indlist[ind]>1:
                banklist.append([indlist[ind], indlist[ind+1]])
        banklist.append([indlist[-1],len(ps)])
        print(banklist)
        for bankrange in banklist:
            sqldict["Name"].append(ps[bankrange[0]].text.strip())
            for bankindex in range(bankrange[0]+1,bankrange[1]):
                if "BP " in ps[bankindex].text or "BP:" in ps[bankindex].text:
                    sqldict["Address_1"].append(ps[bankindex].text.strip())
                if "Tél" in ps[bankindex].text or "Tél:" in ps[bankindex].text or "tél:" in ps[bankindex].text or "Téléphone:" in ps[bankindex].text or "Téléphone :" in ps[bankindex].text:
                    phonefax = ps[bankindex].text.strip().lower().split("fax")
                    sqldict["Phone"].append(phonefax[0])
                    if len(phonefax)>1:
                        sqldict["Fax"].append(phonefax[1])
                if "Site Web:" in ps[bankindex].text or "Site web:" in ps[bankindex].text or "Site web :" in ps[bankindex].text:
                    sqldict["Website"].append(ps[bankindex].text.strip())
                if "Courriel:" in ps[bankindex].text:
                    sqldict["Email"].append(ps[bankindex].text.strip())
                if "SWIFT" in ps[bankindex].text.upper():
                    sqldict["BIC SWIFT Code"].append(ps[bankindex].text.strip())
                
                
                #print(ps[bankindex])
            #print("**********")
            for key in sqldict.keys():
                if len(sqldict["Name"])>len(sqldict[key]):
                    sqldict[key].append("")
    if regdict[reg][0] == "case 2":
        pass
        

    
           # for key in sqldict.keys():
            #    if len(sqldict['Name'])>len(sqldict[key]):
             #       sqldict[key].append("")  
         
df=pd.DataFrame(sqldict,columns=['bvdid', 'priority', 'ListLabel', 'Typology', 'EntryType', 'Name', 'InternalID_1', 'InternalID_1_type', 'InternalID_2', 
  'InternalID_2_type', 'InternalID_3', 'InternalID_3_type', 'CoType', 'License_Type', 'Address_1', 'Address_2', 'City', 
  'Zip', 'Cntry', 'Phone', 'Fax', 'Website', 'Email', 'RegulationType', 'RegulationTypeCode', 'RegulationDate', 'CancellationDate', 
  'RegCtry', 'RegCode', 'ListCode', 'ListLanguage', 'ListValidityDate', 'ListName', 'ListProcessDate', 'LEI Code', 'BIC SWIFT Code', 'Name - Mother Company',
  'Address_1 - Mother company', 'Address_2 -  Mother company', 'City - Mother company', 'Zip - Mother company', 'Cntry - Mother company', 
  'Phone - Mother company', 'Check'])

df.to_excel(writer,'SQL', index=False)
writer.save()
writer.close()
sleep(3)

driver.quit()

endtime=datetime.datetime.now()
difference=endtime-now
difference=difference.total_seconds()
file = open(filename.replace('data', 'time').replace('xlsx','txt'),'w') 
file.write("Start: {} \nEnd: {} \nTotal: {} hours, {} minutes and {} seconds.".format(str(now)[:-7], str(endtime)[:-7], int(difference//3600),int(difference%3600)//60,int(difference%3600)%60))
file.close()


    
    

running  BI BRB webscaper
working with BI BRB 1


AttributeError: 'NoneType' object has no attribute 'find_all'